In [0]:
# ============================================================
# H2 THERMODYNAMIC FEATURES
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Load H2 train/test datasets
# ------------------------------------------------------------

hydrogen_train = (
    spark.table("ml_perovskites.ml_hydrogen_train")
    .toPandas()
)

hydrogen_test = (
    spark.table("ml_perovskites.ml_hydrogen_test")
    .toPandas()
)

print("=" * 70)
print("H2 DATASETS LOADED")
print("=" * 70)

print("\nTraining dataset:")
print(hydrogen_train.shape)

print("\nTest dataset:")
print(hydrogen_test.shape)

print("\nTraining columns:")
print(hydrogen_train.columns.tolist())

print("\nTest columns:")
print(hydrogen_test.columns.tolist())

In [0]:
# ============================================================
# CMD 02 — LOAD ORIGINAL H2 EXPERIMENTAL METADATA
# ============================================================

# Load original source table
db_ml = (
    spark.table("ml_perovskites.db_ml")
    .toPandas()
)

# ------------------------------------------------------------
# Columns relevant to H2 experimental conditions
# ------------------------------------------------------------

h2_metadata_columns = [
    "Compound",
    "hydrogen_production_rate",
    "hydrogen_production_rate_uom",
    "incident_light_cut_off",
    "incident_light_cut_off_uom",
    "particle_size",
    "particle_size_uom",
    "radiation_source_intensity",
    "radiation_source_intensity_uom",
    "specific_surface_area",
    "specific_surface_area_uom",
    "catalyst_amount",
    "catalyst_amount_uom",
    "co_catalyst",
    "co_catalyst_type",
    "reaction_type",
    "reaction_solution",
    "reaction_solution_concentration",
    "carbon_source",
    "sacrificial_agent",
    "sacrificial_agent_source",
    "preparation_method",
    "calcination_temperature",
    "calcination_temperature_uom",
    "calcination_time",
    "calcination_time_uom",
    "doi_reference"
]

missing_metadata = [
    c for c in h2_metadata_columns
    if c not in db_ml.columns
]

if missing_metadata:
    raise RuntimeError(
        f"Missing source columns: {missing_metadata}"
    )

h2_metadata = db_ml[
    h2_metadata_columns
].copy()

print("=" * 70)
print("ORIGINAL H2 METADATA")
print("=" * 70)

print("\nShape:")
print(h2_metadata.shape)

print("\nColumns:")
print(h2_metadata.columns.tolist())

print("\nUnique compounds:")
print(h2_metadata["Compound"].nunique())

print("\nReaction types:")
display(
    h2_metadata["reaction_type"]
    .value_counts(dropna=False)
    .to_frame("count")
)

In [0]:
# ============================================================
# CMD 03 — AUDIT CHEMICAL INPUTS FOR THERMO
# ============================================================

thermo_input_columns = [
    "Compound",
    "reaction_type",
    "reaction_solution",
    "reaction_solution_concentration",
    "carbon_source",
    "sacrificial_agent",
    "sacrificial_agent_source",
    "co_catalyst",
    "co_catalyst_type",
    "catalyst_amount",
    "catalyst_amount_uom",
    "radiation_source_intensity",
    "radiation_source_intensity_uom"
]

for col in thermo_input_columns:

    print("\n" + "=" * 70)
    print(f"{col}")
    print("=" * 70)

    print(
        "Unique values:",
        h2_metadata[col].nunique(dropna=False)
    )

    display(
        h2_metadata[col]
        .value_counts(dropna=False)
        .head(30)
        .to_frame("count")
    )

In [0]:
# ======================================================================
# CMD 03.1 — AUDIT CHEMICAL INPUTS FOR THERMODYNAMIC FEATURE GENERATION
# ======================================================================

thermo_chemical_columns = [
    "reaction_type",
    "reaction_solution",
    "reaction_solution_concentration",
    "carbon_source",
    "sacrificial_agent",
    "sacrificial_agent_source",
    "co_catalyst",
    "co_catalyst_type"
]

display(
    h2_metadata[thermo_chemical_columns]
    .drop_duplicates()
    .sort_values(thermo_chemical_columns)
)

In [0]:
thermo_species_columns = [
    "reaction_solution",
    "carbon_source",
    "sacrificial_agent",
    "sacrificial_agent_source",
    "co_catalyst",
    "co_catalyst_type"
]

for col in thermo_species_columns:
    print("\n" + "=" * 70)
    print(col)
    print("=" * 70)
    
    values = (
        h2_metadata[col]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", np.nan)
        .dropna()
        .unique()
    )
    
    for value in sorted(values):
        print(value)

In [0]:
# ======================================================================
# CMD 03.2A — INSTALL THERMO PACKAGE
# ======================================================================

%pip install thermo

In [0]:
from thermo import Chemical

thermo_test_species = [
    "H2O",
    "methanol",
    "ethanol",
    "glycerol",
    "formaldehyde",
    "formic acid",
    "HBr",
    "HI",
    "H3PO2",
    "NaOH",
    "Na2SO4",
    "Na2SO3"
]

thermo_test_results = []

for species in thermo_test_species:
    try:
        chemical = Chemical(species)

        thermo_test_results.append({
            "input": species,
            "CAS": chemical.CAS,
            "formula": chemical.formula,
            "MW": chemical.MW,
            "Tm": chemical.Tm,
            "Tb": chemical.Tb,
            "Hfus": chemical.Hfus,
            "Hvap": chemical.Hvap,
        })

    except Exception as e:
        thermo_test_results.append({
            "input": species,
            "CAS": None,
            "formula": None,
            "MW": None,
            "Tm": None,
            "Tb": None,
            "Hfus": None,
            "Hvap": None,
        })

thermo_test_df = pd.DataFrame(thermo_test_results)

display(thermo_test_df)

In [0]:
# ======================================================================
# CMD 03.3 — DEFINE VALID THERMODYNAMIC SPECIES
# ======================================================================

thermo_species_map = {
    "Water": "H2O",
    "methanol": "methanol",
    "ethanol": "ethanol",
    "glycerol": "glycerol",
    "formaldehyde": "formaldehyde",
    "formic acid": "formic acid",
    "HBr": "HBr",
    "HI": "HI",
    "H3PO2": "H3PO2",
    "NaOH": "NaOH",
    "Na2SO4": "Na2SO4",
    "Na2SO3": "Na2SO3",
    "lactic acid": "lactic acid",
    "TEOA": "triethanolamine",
    "TEOS": "tetraethyl orthosilicate",
}

print("Defined thermodynamic species:", len(thermo_species_map))

display(
    pd.DataFrame(
        list(thermo_species_map.items()),
        columns=["dataset_name", "thermo_name"]
    )
)

In [0]:
# ======================================================================
# CMD 03.4 — VALIDATE THERMODYNAMIC SPECIES MAPPING
# ======================================================================

thermo_validation = []

for dataset_name, thermo_name in thermo_species_map.items():
    try:
        chemical = Chemical(thermo_name)

        thermo_validation.append({
            "dataset_name": dataset_name,
            "thermo_name": thermo_name,
            "status": "valid",
            "CAS": chemical.CAS,
            "formula": chemical.formula,
            "MW": chemical.MW,
            "Tm": chemical.Tm,
            "Tb": chemical.Tb,
            "Hfus": chemical.Hfus,
            "Hvap": chemical.Hvap
        })

    except Exception as e:
        thermo_validation.append({
            "dataset_name": dataset_name,
            "thermo_name": thermo_name,
            "status": "invalid",
            "CAS": None,
            "formula": None,
            "MW": None,
            "Tm": None,
            "Tb": None,
            "Hfus": None,
            "Hvap": None
        })

thermo_validation_df = pd.DataFrame(thermo_validation)

display(thermo_validation_df)

print(
    "\nValid species:",
    (thermo_validation_df["status"] == "valid").sum(),
    "/",
    len(thermo_validation_df)
)

In [0]:
# ======================================================================
# CMD 03.5 — GENERATE THERMODYNAMIC SPECIES DESCRIPTORS
# ======================================================================

valid_thermo_species = thermo_validation_df[
    thermo_validation_df["status"] == "valid"
].copy()

thermo_feature_columns = [
    "MW",
    "Tm",
    "Tb",
    "Hfus",
    "Hvap"
]

thermo_species_features = (
    valid_thermo_species[
        ["dataset_name", "thermo_name"] + thermo_feature_columns
    ]
    .copy()
)

thermo_species_features = thermo_species_features.rename(
    columns={
        "MW": "thermo_MW",
        "Tm": "thermo_Tm",
        "Tb": "thermo_Tb",
        "Hfus": "thermo_Hfus",
        "Hvap": "thermo_Hvap"
    }
)

display(thermo_species_features)

print(
    "\nValidated species:",
    len(thermo_species_features)
)

print(
    "Unresolved species:",
    sorted(
        set(thermo_species_map.keys())
        - set(thermo_species_features["dataset_name"])
    )
)

In [0]:
# ======================================================================
# CMD 03.6 — MAP THERMODYNAMIC PROPERTIES TO H₂ EXPERIMENTS
# ======================================================================

thermo_feature_columns = [
    "thermo_MW",
    "thermo_Tm",
    "thermo_Tb",
    "thermo_Hfus",
    "thermo_Hvap"
]

thermo_lookup = (
    thermo_species_features
    .set_index("dataset_name")[thermo_feature_columns]
    .to_dict("index")
)

def get_thermo_features(value):
    if pd.isna(value):
        return {col: np.nan for col in thermo_feature_columns}

    value = str(value).strip()

    if value in thermo_lookup:
        return thermo_lookup[value]

    return {col: np.nan for col in thermo_feature_columns}


h2_thermo = h2_metadata.copy()

thermo_mapped = (
    h2_thermo["reaction_solution"]
    .apply(get_thermo_features)
    .apply(pd.Series)
)

h2_thermo = pd.concat(
    [h2_thermo, thermo_mapped],
    axis=1
)

print("H₂ thermodynamic dataset shape:", h2_thermo.shape)

display(
    h2_thermo[
        [
            "Compound",
            "reaction_solution",
            "thermo_MW",
            "thermo_Tm",
            "thermo_Tb",
            "thermo_Hfus",
            "thermo_Hvap"
        ]
    ].head(20)
)

In [0]:
# ======================================================================
# CMD 03.7 — PARSE MULTI-SPECIES REACTION SOLUTIONS
# ======================================================================

def parse_reaction_species(solution):
    if pd.isna(solution):
        return []

    solution = str(solution).strip()

    if not solution or solution == "-":
        return []

    return [
        item.strip()
        for item in solution.split("/")
        if item.strip()
    ]


h2_thermo["reaction_species"] = (
    h2_thermo["reaction_solution"]
    .apply(parse_reaction_species)
)

h2_thermo["n_reaction_species"] = (
    h2_thermo["reaction_species"]
    .apply(len)
)

display(
    h2_thermo[
        [
            "Compound",
            "reaction_solution",
            "reaction_species",
            "n_reaction_species"
        ]
    ].head(30)
)

In [0]:
# ======================================================================
# CMD 03.8 — CALCULATE MIXTURE-LEVEL THERMODYNAMIC DESCRIPTORS
# ======================================================================

def calculate_mixture_thermo(species_list):
    if not species_list:
        return {
            "thermo_mean_MW": np.nan,
            "thermo_mean_Tm": np.nan,
            "thermo_mean_Tb": np.nan,
            "thermo_mean_Hfus": np.nan,
            "thermo_mean_Hvap": np.nan,
            "thermo_species_valid_fraction": np.nan
        }

    valid_features = []

    for species in species_list:
        if species in thermo_lookup:
            valid_features.append(thermo_lookup[species])

    if not valid_features:
        return {
            "thermo_mean_MW": np.nan,
            "thermo_mean_Tm": np.nan,
            "thermo_mean_Tb": np.nan,
            "thermo_mean_Hfus": np.nan,
            "thermo_mean_Hvap": np.nan,
            "thermo_species_valid_fraction": 0.0
        }

    feature_df = pd.DataFrame(valid_features)

    return {
        "thermo_mean_MW": feature_df["thermo_MW"].mean(),
        "thermo_mean_Tm": feature_df["thermo_Tm"].mean(),
        "thermo_mean_Tb": feature_df["thermo_Tb"].mean(),
        "thermo_mean_Hfus": feature_df["thermo_Hfus"].mean(),
        "thermo_mean_Hvap": feature_df["thermo_Hvap"].mean(),
        "thermo_species_valid_fraction": (
            len(valid_features) / len(species_list)
        )
    }


mixture_thermo = (
    h2_thermo["reaction_species"]
    .apply(calculate_mixture_thermo)
    .apply(pd.Series)
)

h2_thermo = pd.concat(
    [h2_thermo, mixture_thermo],
    axis=1
)

display(
    h2_thermo[
        [
            "Compound",
            "reaction_solution",
            "reaction_species",
            "thermo_mean_MW",
            "thermo_mean_Tm",
            "thermo_mean_Tb",
            "thermo_mean_Hfus",
            "thermo_mean_Hvap",
            "thermo_species_valid_fraction"
        ]
    ].head(30)
)

In [0]:
# ======================================================================
# CMD 03.9 — AUDIT THERMODYNAMIC FEATURE COVERAGE
# ======================================================================

thermo_feature_audit = pd.DataFrame({
    "feature": [
        "thermo_mean_MW",
        "thermo_mean_Tm",
        "thermo_mean_Tb",
        "thermo_mean_Hfus",
        "thermo_mean_Hvap",
        "thermo_species_valid_fraction"
    ],
    "non_missing": [
        h2_thermo["thermo_mean_MW"].notna().sum(),
        h2_thermo["thermo_mean_Tm"].notna().sum(),
        h2_thermo["thermo_mean_Tb"].notna().sum(),
        h2_thermo["thermo_mean_Hfus"].notna().sum(),
        h2_thermo["thermo_mean_Hvap"].notna().sum(),
        h2_thermo["thermo_species_valid_fraction"].notna().sum()
    ],
    "missing": [
        h2_thermo["thermo_mean_MW"].isna().sum(),
        h2_thermo["thermo_mean_Tm"].isna().sum(),
        h2_thermo["thermo_mean_Tb"].isna().sum(),
        h2_thermo["thermo_mean_Hfus"].isna().sum(),
        h2_thermo["thermo_mean_Hvap"].isna().sum(),
        h2_thermo["thermo_species_valid_fraction"].isna().sum()
    ]
})

thermo_feature_audit["coverage_pct"] = (
    thermo_feature_audit["non_missing"]
    / len(h2_thermo)
    * 100
)

display(thermo_feature_audit)

In [0]:
# ======================================================================
# CMD 03.10 — AUDIT VALID THERMODYNAMIC SPECIES FRACTION
# ======================================================================

print("Distribution of valid thermodynamic species fraction:")
display(
    h2_thermo["thermo_species_valid_fraction"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("count")
)

print("\nSummary:")
display(
    h2_thermo["thermo_species_valid_fraction"]
    .describe()
    .to_frame("value")
)

print("\nExperiments with incomplete species resolution:")

incomplete_thermo = h2_thermo[
    h2_thermo["thermo_species_valid_fraction"] < 1.0
][
    [
        "Compound",
        "reaction_solution",
        "reaction_species",
        "thermo_species_valid_fraction"
    ]
]

display(incomplete_thermo)

In [0]:
# ======================================================================
# CMD 03.11 — IDENTIFY UNRESOLVED THERMODYNAMIC SPECIES
# ======================================================================

incomplete_species = []

for _, row in h2_thermo[
    h2_thermo["thermo_species_valid_fraction"] < 1.0
].iterrows():

    for species in row["reaction_species"]:
        if species not in thermo_lookup:
            incomplete_species.append(species)

unresolved_species_counts = (
    pd.Series(incomplete_species)
    .value_counts()
    .rename_axis("unresolved_species")
    .reset_index(name="count")
)

display(unresolved_species_counts)

In [0]:
# ======================================================================
# CMD 03.12 — EXPAND THERMODYNAMIC SPECIES ALIASES
# ======================================================================

thermo_species_map.update({
    "Formic Acid": "formic acid",
    "Lactic acid": "lactic acid",
    "DEA": "diethanolamine",
    "Na2S": "sodium sulfide",
    "H3PO2": "hypophosphorous acid"
})

print("Updated thermodynamic species mappings:")
display(
    pd.DataFrame(
        list(thermo_species_map.items()),
        columns=["dataset_name", "thermo_name"]
    )
)

In [0]:
# ======================================================================
# CMD 03.13 — VALIDATE EXPANDED THERMODYNAMIC SPECIES ALIASES
# ======================================================================

alias_validation = []

for dataset_name in [
    "Na2S",
    "H3PO2",
    "Formic Acid",
    "Lactic acid",
    "DEA"
]:
    thermo_name = thermo_species_map[dataset_name]

    try:
        chemical = Chemical(thermo_name)

        alias_validation.append({
            "dataset_name": dataset_name,
            "thermo_name": thermo_name,
            "status": "valid",
            "CAS": chemical.CAS,
            "formula": chemical.formula,
            "MW": chemical.MW,
            "Tm": chemical.Tm,
            "Tb": chemical.Tb,
            "Hfus": chemical.Hfus,
            "Hvap": chemical.Hvap
        })

    except Exception:
        alias_validation.append({
            "dataset_name": dataset_name,
            "thermo_name": thermo_name,
            "status": "invalid",
            "CAS": None,
            "formula": None,
            "MW": None,
            "Tm": None,
            "Tb": None,
            "Hfus": None,
            "Hvap": None
        })

alias_validation_df = pd.DataFrame(alias_validation)

display(alias_validation_df)

In [0]:
# ======================================================================
# CMD 03.14 — REBUILD THERMODYNAMIC SPECIES LOOKUP
# ======================================================================

all_thermo_species = pd.concat(
    [
        thermo_validation_df[
            thermo_validation_df["status"] == "valid"
        ],
        alias_validation_df[
            alias_validation_df["status"] == "valid"
        ]
    ],
    ignore_index=True
)

all_thermo_species = (
    all_thermo_species
    .drop_duplicates(subset=["dataset_name"], keep="last")
    .reset_index(drop=True)
)

thermo_feature_columns = [
    "thermo_MW",
    "thermo_Tm",
    "thermo_Tb",
    "thermo_Hfus",
    "thermo_Hvap"
]

all_thermo_species = all_thermo_species.rename(
    columns={
        "MW": "thermo_MW",
        "Tm": "thermo_Tm",
        "Tb": "thermo_Tb",
        "Hfus": "thermo_Hfus",
        "Hvap": "thermo_Hvap"
    }
)

thermo_lookup = (
    all_thermo_species
    .set_index("dataset_name")[thermo_feature_columns]
    .to_dict("index")
)

print("Total validated thermodynamic species:", len(all_thermo_species))

display(
    all_thermo_species[
        ["dataset_name", "thermo_name"] + thermo_feature_columns
    ]
)

In [0]:
# ======================================================================
# CMD 03.15 — RECALCULATE THERMODYNAMIC DESCRIPTORS WITH COMPLETE SPECIES MAPPING
# ======================================================================

def calculate_mixture_thermo_updated(species_list):
    if not species_list:
        return {
            "thermo_mean_MW": np.nan,
            "thermo_mean_Tm": np.nan,
            "thermo_mean_Tb": np.nan,
            "thermo_mean_Hfus": np.nan,
            "thermo_mean_Hvap": np.nan,
            "thermo_species_valid_fraction": np.nan
        }

    valid_features = []

    for species in species_list:
        if species in thermo_lookup:
            valid_features.append(thermo_lookup[species])

    result = {
        "thermo_mean_MW": np.nan,
        "thermo_mean_Tm": np.nan,
        "thermo_mean_Tb": np.nan,
        "thermo_mean_Hfus": np.nan,
        "thermo_mean_Hvap": np.nan,
        "thermo_species_valid_fraction": (
            len(valid_features) / len(species_list)
        )
    }

    if valid_features:
        feature_df = pd.DataFrame(valid_features)

        result["thermo_mean_MW"] = feature_df["thermo_MW"].mean()
        result["thermo_mean_Tm"] = feature_df["thermo_Tm"].mean()
        result["thermo_mean_Tb"] = feature_df["thermo_Tb"].mean()
        result["thermo_mean_Hfus"] = feature_df["thermo_Hfus"].mean()
        result["thermo_mean_Hvap"] = feature_df["thermo_Hvap"].mean()

    return result


thermo_recalculated = (
    h2_thermo["reaction_species"]
    .apply(calculate_mixture_thermo_updated)
    .apply(pd.Series)
)

thermo_feature_columns = [
    "thermo_mean_MW",
    "thermo_mean_Tm",
    "thermo_mean_Tb",
    "thermo_mean_Hfus",
    "thermo_mean_Hvap",
    "thermo_species_valid_fraction"
]

h2_thermo[thermo_feature_columns] = thermo_recalculated[
    thermo_feature_columns
]

print("Thermodynamic descriptors recalculated.")

display(
    h2_thermo[
        [
            "Compound",
            "reaction_solution",
            "reaction_species"
        ] + thermo_feature_columns
    ].head(30)
)

In [0]:
# ======================================================================
# CMD 03.16 — VERIFY COMPLETE THERMODYNAMIC SPECIES COVERAGE
# ======================================================================

complete_count = (
    h2_thermo["thermo_species_valid_fraction"] == 1.0
).sum()

incomplete_count = (
    h2_thermo["thermo_species_valid_fraction"] < 1.0
).sum()

coverage_summary = pd.DataFrame({
    "metric": [
        "Total experiments",
        "Complete species resolution",
        "Incomplete species resolution",
        "Complete resolution (%)"
    ],
    "value": [
        len(h2_thermo),
        complete_count,
        incomplete_count,
        h2_thermo["thermo_species_valid_fraction"].eq(1.0).mean() * 100
    ]
})

display(coverage_summary)

if incomplete_count > 0:
    print("\nIncomplete experiments:")

    display(
        h2_thermo[
            h2_thermo["thermo_species_valid_fraction"] < 1.0
        ][
            [
                "Compound",
                "reaction_solution",
                "reaction_species",
                "thermo_species_valid_fraction"
            ]
        ]
    )
else:
    print("\nAll experiments have complete thermodynamic species resolution.")

In [0]:
# ======================================================================
# CMD 03.17 — AUDIT THERMODYNAMIC FEATURE MISSINGNESS
# ======================================================================

thermo_features = [
    "thermo_mean_MW",
    "thermo_mean_Tm",
    "thermo_mean_Tb",
    "thermo_mean_Hfus",
    "thermo_mean_Hvap"
]

thermo_missingness = pd.DataFrame({
    "feature": thermo_features,
    "non_missing": [
        h2_thermo[col].notna().sum()
        for col in thermo_features
    ],
    "missing": [
        h2_thermo[col].isna().sum()
        for col in thermo_features
    ]
})

thermo_missingness["coverage_pct"] = (
    thermo_missingness["non_missing"]
    / len(h2_thermo)
    * 100
)

display(thermo_missingness)

In [0]:
# ======================================================================
# CMD 03.18 — ATTACH H₂ TARGET TO THERMODYNAMIC DATASET
# ======================================================================

h2_thermo["hydrogen_target"] = (
    h2_metadata["hydrogen_production_rate"]
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False)
)

h2_thermo["hydrogen_target"] = pd.to_numeric(
    h2_thermo["hydrogen_target"],
    errors="coerce"
)

print("Thermodynamic records:", len(h2_thermo))
print(
    "Valid H₂ targets:",
    h2_thermo["hydrogen_target"].notna().sum()
)

assert len(h2_thermo) == 440
assert h2_thermo["hydrogen_target"].notna().all()

display(
    h2_thermo[
        [
            "Compound",
            "hydrogen_target",
            "reaction_solution"
        ]
    ].head(10)
)


In [0]:
# ======================================================================
# CMD 03.19 — CREATE UNIQUE THERMODYNAMIC MODELING KEY
# ======================================================================

thermo_features = [
    "thermo_mean_MW",
    "thermo_mean_Tm",
    "thermo_mean_Tb",
    "thermo_mean_Hfus",
    "thermo_mean_Hvap"
]

thermo_model_key = (
    h2_thermo[
        [
            "Compound",
            "doi_reference",
            "hydrogen_target"
        ] + thermo_features
    ]
    .drop_duplicates(
        subset=[
            "Compound",
            "doi_reference",
            "hydrogen_target"
        ],
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    "Unique thermodynamic modeling keys:",
    len(thermo_model_key)
)

print(
    "Thermodynamic feature rows:",
    len(thermo_model_key)
)

display(
    thermo_model_key.head(10)
)

In [0]:
# ======================================================================
# CMD 03.20 — MERGE THERMODYNAMIC FEATURES INTO H₂ MODELING DATA
# ======================================================================

hydrogen_train_thermo = hydrogen_train.merge(
    thermo_model_key,
    on=[
        "Compound",
        "doi_reference",
        "hydrogen_target"
    ],
    how="left",
    validate="many_to_one"
)

hydrogen_test_thermo = hydrogen_test.merge(
    thermo_model_key,
    on=[
        "Compound",
        "doi_reference",
        "hydrogen_target"
    ],
    how="left",
    validate="many_to_one"
)

print("H₂ train with thermodynamic features:", hydrogen_train_thermo.shape)
print("H₂ test with thermodynamic features:", hydrogen_test_thermo.shape)

print("\nTrain thermodynamic coverage:")
display(
    hydrogen_train_thermo[thermo_features]
    .notna()
    .sum()
    .to_frame("non_missing")
)

print("\nTest thermodynamic coverage:")
display(
    hydrogen_test_thermo[thermo_features]
    .notna()
    .sum()
    .to_frame("non_missing")
)

In [0]:
# ======================================================================
# CMD 03.21 — AUDIT COMBINED H₂ FEATURE SET
# ======================================================================

combined_feature_columns = [
    col for col in hydrogen_train_thermo.columns
    if col not in [
        "Compound",
        "doi_reference",
        "hydrogen_target"
    ]
]

print(
    "Total modeling features:",
    len(combined_feature_columns)
)

print("\nFeature groups:")
print("Composition descriptors:", 43)
print("Thermodynamic descriptors:", len(thermo_features))
print("Total:", 43 + len(thermo_features))

print("\nTrain shape:", hydrogen_train_thermo.shape)
print("Test shape:", hydrogen_test_thermo.shape)

print("\nMissing values — train:")
display(
    hydrogen_train_thermo[combined_feature_columns]
    .isna()
    .sum()
    .to_frame("missing")
)

print("\nMissing values — test:")
display(
    hydrogen_test_thermo[combined_feature_columns]
    .isna()
    .sum()
    .to_frame("missing")
)

In [0]:
# ======================================================================
# CMD 03.22 — AUDIT CORRELATION IN COMBINED H₂ FEATURES
# ======================================================================

correlation_matrix = (
    hydrogen_train_thermo[combined_feature_columns]
    .corr(method="pearson")
)

correlation_threshold = 0.95

high_correlation_pairs = []

for i in range(len(combined_feature_columns)):
    for j in range(i + 1, len(combined_feature_columns)):

        correlation = correlation_matrix.iloc[i, j]

        if abs(correlation) >= correlation_threshold:
            high_correlation_pairs.append({
                "feature_1": combined_feature_columns[i],
                "feature_2": combined_feature_columns[j],
                "pearson_r": correlation,
                "abs_pearson_r": abs(correlation)
            })

print(
    "Highly correlated feature pairs (|r| >= 0.95):",
    len(high_correlation_pairs)
)

if high_correlation_pairs:
    high_correlation_pairs_df = pd.DataFrame(
        high_correlation_pairs
    ).sort_values(
        "abs_pearson_r",
        ascending=False
    ).reset_index(drop=True)

    display(high_correlation_pairs_df)
else:
    print(
        "No feature pairs meet the |r| >= 0.95 threshold."
    )

In [0]:
# ======================================================================
# CMD 03.23 — PREPARE X/y MATRICES FOR MODELING
# ======================================================================

X_train_h2 = hydrogen_train_thermo[
    combined_feature_columns
].copy()

X_test_h2 = hydrogen_test_thermo[
    combined_feature_columns
].copy()

y_train_h2 = hydrogen_train_thermo[
    "hydrogen_target"
].copy()

y_test_h2 = hydrogen_test_thermo[
    "hydrogen_target"
].copy()

print("X_train shape:", X_train_h2.shape)
print("X_test shape :", X_test_h2.shape)
print("y_train shape:", y_train_h2.shape)
print("y_test shape :", y_test_h2.shape)

print(
    "\nMissing values in X_train:",
    X_train_h2.isna().sum().sum()
)

print(
    "Missing values in X_test :",
    X_test_h2.isna().sum().sum()
)

In [0]:
# ======================================================================
# CMD 03.24 — FIT STANDARD SCALER ON TRAINING DATA
# ======================================================================

from sklearn.preprocessing import StandardScaler

scaler_h2 = StandardScaler()

X_train_h2_scaled = scaler_h2.fit_transform(
    X_train_h2
)

X_test_h2_scaled = scaler_h2.transform(
    X_test_h2
)

print("Scaled X_train shape:", X_train_h2_scaled.shape)
print("Scaled X_test shape :", X_test_h2_scaled.shape)

print(
    "\nTraining feature mean range:",
    X_train_h2_scaled.mean(axis=0).min(),
    "to",
    X_train_h2_scaled.mean(axis=0).max()
)

print(
    "Training feature std range:",
    X_train_h2_scaled.std(axis=0).min(),
    "to",
    X_train_h2_scaled.std(axis=0).max()
)

In [0]:
# ======================================================================
# CMD 03.25 — PREPARE LOG-TRANSFORMED H₂ TARGET
# ======================================================================

y_train_h2_log = np.log1p(y_train_h2)
y_test_h2_log = np.log1p(y_test_h2)

print("Original target range:")
print("Train:", y_train_h2.min(), "to", y_train_h2.max())
print("Test :", y_test_h2.min(), "to", y_test_h2.max())

print("\nLog-transformed target range:")
print("Train:", y_train_h2_log.min(), "to", y_train_h2_log.max())
print("Test :", y_test_h2_log.min(), "to", y_test_h2_log.max())

print(
    "\nTraining target mean:",
    y_train_h2_log.mean()
)

print(
    "Training target standard deviation:",
    y_train_h2_log.std()
)